In [1]:
from pathlib import Path
import sqlite3
from html import escape
from IPython.display import display, HTML, Markdown

# Review only: this notebook never writes to the populated database.
here = Path.cwd().resolve()
project = next(p for p in (here, *here.parents) if (p / 'data/analysis.sqlite').is_file())
DB_PATH = project / 'data/analysis.sqlite'
db = sqlite3.connect(DB_PATH.as_uri() + '?mode=ro', uri=True)

def show_query(sql):
    cursor = db.execute(sql)
    headers = ''.join(f'<th>{escape(column[0])}</th>' for column in cursor.description)
    rows = ''.join('<tr>' + ''.join(f'<td>{escape(str(value))}</td>' for value in row) + '</tr>' for row in cursor)
    display(HTML(f'<table><thead><tr>{headers}</tr></thead><tbody>{rows}</tbody></table>'))

display(Markdown('# Person clips: population review'))
print(f'Database: {DB_PATH}')
display(Markdown('### 1. Populated tables — five consecutive analyzed frames per person'))
show_query("""
SELECT 'Clip' AS table_name, COUNT(*) AS rows FROM Clip
UNION ALL SELECT 'ClipPerson', COUNT(*) FROM ClipPerson
UNION ALL SELECT 'ClipGeneration (finalized videos)', COUNT(*) FROM ClipGeneration
UNION ALL SELECT 'Activity (existing predictions)', COUNT(*) FROM Activity
UNION ALL SELECT 'Engagement (existing scores)', COUNT(*) FROM Engagement
""")


# Person clips: population review

Database: /mnt/storage/Projects/ai_for_good_2026/data/analysis.sqlite


### 1. Populated tables — five consecutive analyzed frames per person

table_name,rows
Clip,1988
ClipPerson,3893
ClipGeneration (finalized videos),12
Activity (existing predictions),599
Engagement (existing scores),50514


In [2]:
display(Markdown('### 2. Clip population by source video'))
show_query("""
WITH segments AS (
    SELECT video_id, COUNT(*) AS clips,
           SUM(end_time_seconds - start_time_seconds) AS covered_seconds
    FROM Clip GROUP BY video_id
), members AS (
    SELECT c.video_id, COUNT(*) AS memberships, COUNT(DISTINCT cp.person_id) AS people
    FROM Clip c JOIN ClipPerson cp USING (clip_id) GROUP BY c.video_id
)
SELECT v.video_id, v.filename AS source_video,
       COALESCE(s.clips, 0) AS clips, COALESCE(m.memberships, 0) AS memberships,
       COALESCE(m.people, 0) AS people,
       ROUND(COALESCE(s.covered_seconds, 0), 2) AS covered_seconds,
       g.persistence_frames
FROM Video v LEFT JOIN segments s USING (video_id)
LEFT JOIN members m USING (video_id)
LEFT JOIN ClipGeneration g USING (video_id)
ORDER BY v.video_id
""")


### 2. Clip population by source video

video_id,source_video,clips,memberships,people,covered_seconds,persistence_frames
1,THIS IS ME 2025-FINAL VIDEO-1080p.mp4,168,260,28,814.44,5
2,THIS IS ME Live (Camp2).mp4,911,1894,70,5958.87,5
3,THIS IS ME-camp2 montage-FINAL.mp4,177,264,39,995.2,5
4,THIS IS ME-final show-camp1.mp4,635,1311,50,4168.46,5
5,THIS IS ME-intro.mp4,48,96,27,108.78,5
6,this is me-songwriting-camp2-deshaun.mp4,22,24,4,218.22,5
7,this is me-songwriting-camp2-reese.mp4,1,1,1,152.03,5
8,this is me3-tik tok promo-painting.mp4,3,3,2,41.67,5
9,this is me4-tik tok promo-dance.mp4,10,19,5,26.94,5
10,this is me5-tik tok promo-rap.mp4,1,1,1,21.4,5


In [3]:
display(Markdown('### 3. Sample clips and associated people — first clip per video'))
show_query("""
SELECT c.clip_id, c.video_id, ROUND(c.start_time_seconds, 3) AS start_seconds,
       ROUND(c.end_time_seconds, 3) AS end_seconds,
       (SELECT GROUP_CONCAT(person_id, ', ') FROM
           (SELECT person_id FROM ClipPerson WHERE clip_id = c.clip_id ORDER BY person_id)
       ) AS person_ids
FROM Clip c
WHERE c.clip_id = (SELECT c2.clip_id FROM Clip c2 WHERE c2.video_id = c.video_id
                   ORDER BY c2.start_time_seconds LIMIT 1)
ORDER BY c.video_id
""")
assert db.execute('SELECT COUNT(*) FROM Clip').fetchone()[0] > 0
assert db.execute('SELECT COUNT(*) FROM ClipGeneration').fetchone()[0] == db.execute('SELECT COUNT(*) FROM Video').fetchone()[0]
assert db.execute('SELECT COUNT(*) FROM ClipGeneration WHERE persistence_frames != 5').fetchone()[0] == 0
assert not db.execute('SELECT 1 FROM Clip c JOIN Video v USING(video_id) WHERE c.start_time_seconds < 0 OR c.end_time_seconds <= c.start_time_seconds OR c.end_time_seconds > v.duration_seconds LIMIT 1').fetchone()
assert not db.execute('SELECT 1 FROM Clip a JOIN Clip b ON a.video_id = b.video_id AND a.clip_id < b.clip_id AND a.start_time_seconds < b.end_time_seconds AND b.start_time_seconds < a.end_time_seconds LIMIT 1').fetchone()
assert not db.execute('SELECT 1 FROM Clip c WHERE NOT EXISTS (SELECT 1 FROM ClipPerson p WHERE p.clip_id = c.clip_id) LIMIT 1').fetchone()
assert db.execute('PRAGMA foreign_key_check').fetchall() == []
assert db.execute('PRAGMA integrity_check').fetchall() == [('ok',)]
print('Checks passed: all videos finalized at threshold 5; valid, nonoverlapping clips; people linked; database integrity OK.')
db.close()


### 3. Sample clips and associated people — first clip per video

clip_id,video_id,start_seconds,end_seconds,person_ids
1,1,6.215,11.094,Person_0008
169,2,3.629,8.842,"Person_0004, Person_0011, Person_0031"
1080,3,1.168,3.003,"Person_0055, Person_0074"
1257,4,4.296,14.556,Person_0009
1892,5,1.168,3.003,"Person_0055, Person_0074"
1940,6,0.5,6.089,Person_0074
1962,7,0.459,152.486,Person_0001
1963,8,1.418,17.976,Person_0005
1966,9,0.5,3.378,"Person_0007, Person_0079"
1976,10,0.375,21.772,Person_0039


Checks passed: all videos finalized at threshold 5; valid, nonoverlapping clips; people linked; database integrity OK.
